# Semana 7 · Sesión 2: Cinemática de partículas y cuerpos rígidos

**Módulo 2**

## Objetivos de la sesión

1. Describir la posición de puntos con `Point` y obtener su velocidad y su
   aceleración respecto a un marco.
2. Reconocer, en coordenadas polares, de dónde salen las aceleraciones
   centrípeta y de Coriolis.
3. Usar los teoremas de dos puntos y de un punto para la cinemática de cuerpos
   rígidos, como el péndulo físico y el péndulo doble.

## Retomamos

La sesión pasada construimos marcos, vectores y rotaciones, y aprendimos la
regla de oro de `mechanics`: **una derivada temporal siempre es respecto a un
marco**. Al orientar un marco con un ángulo que depende del tiempo, la
velocidad angular salió sola.

Nos faltó algo esencial para hacer cinemática: **dónde** está cada cosa. Un
vector dice una dirección y una magnitud, pero no está pegado a ningún lugar.
Hoy agregamos el objeto que sí lo está: el **punto**.

La celda de abajo reconstruye lo que necesitamos de la sesión 1, para que este
notebook corra por sí solo. Se agregan también NumPy y Matplotlib, para la
gráfica del final.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math

me.init_vprinting()

t = sp.Symbol("t")               # el tiempo de mechanics: sin suposiciones
N = me.ReferenceFrame("N")       # el marco del laboratorio

## `Point`: un lugar en el espacio

Un `me.Point` representa un lugar. Por sí solo no tiene coordenadas: se
**ubica respecto a otro punto** con un vector de posición, igual que un marco
se orienta respecto a otro marco.

| Quieres | Escribe |
|---|---|
| Crear un punto | `O = me.Point("O")` |
| Crear otro ya ubicado | `P = O.locatenew("P", vector)` |
| Ubicar uno que ya existe | `P.set_pos(O, vector)` |
| Su posición respecto a otro | `P.pos_from(O)` |
| Fijar su velocidad en un marco | `O.set_vel(N, vector)` |
| Su velocidad / aceleración | `P.vel(N)`, `P.acc(N)` |

La velocidad también es **respecto a un marco**: la velocidad de un pasajero es
cero respecto al tren y enorme respecto a las vías.

In [ ]:
x, y = me.dynamicsymbols("x y")

O = me.Point("O")
O.set_vel(N, 0)                  # O es el origen fijo del laboratorio

P = O.locatenew("P", x*N.x + y*N.y)

display(Math(r"\mathbf{r}_{P/O} = " + me.vlatex(P.pos_from(O))))
display(Math(r"{}^N\mathbf{v}^P = " + me.vlatex(P.vel(N))))
display(Math(r"{}^N\mathbf{a}^P = " + me.vlatex(P.acc(N))))

## Sin un punto de referencia, no hay velocidad

`P.vel(N)` funcionó porque `mechanics` recorrió la cadena de posiciones hasta
encontrar un punto cuya velocidad en `N` sí conoce —`O`, que fijamos con
`set_vel`— y derivó desde ahí. Si ningún punto de la cadena tiene velocidad
conocida, no hay de dónde partir: la velocidad de un punto aislado no está
definida.

Es el error más común de la semana. Veámoslo a propósito.

In [ ]:
origen_suelto = me.Point("Q")                     # sin set_vel
punto_suelto = origen_suelto.locatenew("R", x*N.x)

try:
    punto_suelto.vel(N)
except ValueError as error:
    print("ValueError -", error)

## Física: coordenadas polares

Una partícula se mueve en el plano, descrita por su distancia $r(t)$ al origen
y su ángulo $\theta(t)$. El truco de `mechanics` es no pelearse con senos y
cosenos: creamos un marco $B$ que **gira con la partícula**, de modo que la
posición es simplemente $r\,\hat{b}_x$. Así $\hat{b}_x$ es la dirección radial
y $\hat{b}_y$ la transversal.

Todo lo demás es la sesión 1: el marco gira con $\dot\theta$, y la derivada en
$N$ usa el teorema de transporte.

In [ ]:
r, theta = me.dynamicsymbols("r theta")

B = me.ReferenceFrame("B")
B.orient_axis(N, N.z, theta)      # B gira con la partícula

particula = O.locatenew("P", r*B.x)

display(Math(r"{}^N\mathbf{v}^P = " + me.vlatex(particula.vel(N))))

In [ ]:
aceleracion = particula.acc(N).express(B)

display(sp.Eq(sp.Symbol("a_r"), aceleracion.dot(B.x)))
display(sp.Eq(sp.Symbol(r"a_\theta"), aceleracion.dot(B.y)))

Cada término de la aceleración tiene nombre propio:

| Término | Nombre | Lo que dice |
|---|---|---|
| $\ddot r$ | Aceleración radial "de a de veras" | La distancia al origen cambia cada vez más rápido |
| $-r\dot\theta^2$ | **Centrípeta** | Girar exige acelerar hacia el centro |
| $r\ddot\theta$ | Tangencial | El giro se acelera |
| $2\dot r\dot\theta$ | **De Coriolis** | Alejarse del centro mientras se gira |

Los libros de mecánica dedican una página de derivadas con regla de la cadena a
esta fórmula. Aquí salió de declarar un marco que gira y una posición de un
solo término.

## TODO en clase 1

Una cuenta se desliza por una varilla horizontal que gira alrededor de un eje
vertical con velocidad angular **constante** $\omega$. La distancia de la cuenta
al eje es $s(t)$, libre.

1. Declara `omega` como símbolo positivo y `s` como `dynamicsymbol`.
2. Crea el marco `varilla`, orientado respecto a `N` alrededor de `N.z` con el
   ángulo $\omega t$ — aquí el ángulo no es un `dynamicsymbol`, sino una función
   explícita del tiempo, y `mechanics` la deriva igual.
3. Ubica el punto `cuenta` a una distancia $s$ de `O`, a lo largo de la varilla,
   y calcula `aceleracion_cuenta` expresada en el marco de la varilla. Deben
   salir $\ddot s - \omega^2 s$ a lo largo de la varilla y $2\omega\dot s$
   perpendicular a ella.
4. Con una masa `m` (positiva), calcula `fuerza_varilla`: la fuerza
   perpendicular que la varilla tiene que ejercer sobre la cuenta,
   $m\,\mathbf{a}\cdot\hat{v}_y$. ¿Por qué hace falta una fuerza lateral si la
   cuenta solo se mueve **a lo largo** de la varilla?

In [ ]:
# TODO en clase: la cuenta en la varilla giratoria
omega = ...
s = ...

varilla = ...

cuenta = ...

aceleracion_cuenta = ...

fuerza_varilla = ...

## Cuerpo rígido: el teorema de dos puntos

En un cuerpo rígido las distancias entre sus puntos no cambian. Eso impone una
relación entre las velocidades de dos puntos $P$ y $Q$ **fijos en el mismo
cuerpo** $A$:

$$\mathbf{v}^Q = \mathbf{v}^P + \boldsymbol{\omega}^A \times \mathbf{r}_{Q/P}$$

$$\mathbf{a}^Q = \mathbf{a}^P + \boldsymbol{\alpha}^A \times \mathbf{r}_{Q/P}
  + \boldsymbol{\omega}^A \times (\boldsymbol{\omega}^A \times \mathbf{r}_{Q/P})$$

En `mechanics` se llaman `Q.v2pt_theory(P, N, A)` y `Q.a2pt_theory(P, N, A)`:
"la velocidad de $Q$ en $N$, sabiendo la de $P$ y que ambos están fijos en
$A$". Calculan **y guardan** el resultado en el punto.

Ejemplo: un **péndulo físico**, una varilla de longitud $L$ que cuelga de un
pivote $O$ y oscila con ángulo $\theta$ respecto a la vertical. Su centro de
masa $G$ está a la mitad.

In [ ]:
L = sp.Symbol("L", positive=True)   # longitud de la varilla

A = me.ReferenceFrame("A")
A.orient_axis(N, N.z, theta)        # la varilla gira con el ángulo theta

G = me.Point("G")
G.set_pos(O, -L/2 * A.y)            # colgando: hacia abajo de A

display(Math(r"{}^N\mathbf{v}^G = " + me.vlatex(G.v2pt_theory(O, N, A))))
display(Math(r"{}^N\mathbf{a}^G = " + me.vlatex(G.a2pt_theory(O, N, A))))

La velocidad es perpendicular a la varilla (a lo largo de $\hat{a}_x$) y vale
$\frac{L}{2}\dot\theta$, como en un movimiento circular. La aceleración tiene
la parte tangencial $\frac{L}{2}\ddot\theta$ y la centrípeta
$\frac{L}{2}\dot\theta^2$, que apunta de $G$ hacia el pivote.

¿Y si no confiamos en el teorema? Derivamos la posición directamente en $N$ y
comparamos. Verificar en lugar de creer, como siempre.

In [ ]:
directa = G.pos_from(O).dt(N)
display((directa - G.vel(N)).simplify())

## El teorema de un punto: algo que se mueve sobre el cuerpo

¿Y si el punto **no** está fijo en el cuerpo, sino que se mueve sobre él? Una
hormiga que camina por la varilla del péndulo, a una distancia $s(t)$ del
pivote. Entonces aparece un término más, la velocidad de la hormiga **vista
desde la varilla**:

$$\mathbf{v}^H = \mathbf{v}^O + \boldsymbol{\omega}^A \times \mathbf{r}_{H/O}
  + {}^A\mathbf{v}^H$$

Es `H.v1pt_theory(O, N, A)`, y pide antes la velocidad de $H$ respecto a la
varilla (`H.set_vel(A, ...)`).

In [ ]:
distancia = me.dynamicsymbols("s")

hormiga = me.Point("H")
hormiga.set_pos(O, -distancia * A.y)
hormiga.set_vel(A, -distancia.diff(t) * A.y)   # camina hacia abajo por la varilla

display(Math(r"{}^N\mathbf{v}^H = " + me.vlatex(hormiga.v1pt_theory(O, N, A))))

Dos términos: $s\dot\theta\,\hat{a}_x$, lo que la varilla arrastra a la hormiga
al girar, y $-\dot s\,\hat{a}_y$, lo que la hormiga camina por su cuenta. Es la
misma estructura que la cuenta en la varilla giratoria.

## Física: el péndulo doble

Un péndulo doble: una varilla de longitud $L_1$ cuelga del pivote $O$ con
ángulo $\theta_1$, y de su extremo $P_1$ cuelga otra de longitud $L_2$ con
ángulo $\theta_2$. Los dos ángulos se miden **respecto a la vertical**, así que
los dos marcos se orientan respecto a `N`.

La celda de abajo arma los marcos y los puntos. Las velocidades te tocan a ti.

In [ ]:
theta1, theta2 = me.dynamicsymbols("theta1 theta2")
L1, L2 = sp.symbols("L1 L2", positive=True)

A1 = me.ReferenceFrame("A1")
A1.orient_axis(N, N.z, theta1)
A2 = me.ReferenceFrame("A2")
A2.orient_axis(N, N.z, theta2)

P1 = me.Point("P1")
P1.set_pos(O, -L1 * A1.y)
P2 = me.Point("P2")
P2.set_pos(P1, -L2 * A2.y)

display(Math(r"\mathbf{r}_{P_2/O} = " + me.vlatex(P2.pos_from(O).express(N))))

## TODO en clase 2

1. Calcula `velocidad_1`, la velocidad de $P_1$ en `N`, con `v2pt_theory`:
   $O$ y $P_1$ están fijos en la primera varilla, `A1`.
2. Calcula `velocidad_2`, la de $P_2$, encadenando: $P_1$ y $P_2$ están fijos
   en la segunda varilla, `A2`, y la velocidad de $P_1$ ya la conoces.
3. Comprueba contra la derivada directa de `P2.pos_from(O)` en `N`.
4. Calcula `rapidez_cuadrada_2` $= \mathbf{v}_2 \cdot \mathbf{v}_2$ y
   simplifícala con `sp.trigsimp(sp.expand(...))`. Debe salir

   $$L_1^2\dot\theta_1^2 + L_2^2\dot\theta_2^2
     + 2L_1L_2\dot\theta_1\dot\theta_2\cos(\theta_1 - \theta_2)$$

   Guárdala bien: la semana que viene es la pieza central de la energía
   cinética del péndulo doble.

In [ ]:
# TODO en clase: velocidades del péndulo doble
velocidad_1 = ...

velocidad_2 = ...

rapidez_cuadrada_2 = ...

## De lo simbólico a la gráfica

¿Cómo se mueve **de verdad** un péndulo doble? Todavía no lo sabemos: eso
requiere ecuaciones de movimiento, y es la semana 8. Lo que sí podemos hacer
hoy es **imponer** un movimiento —dos oscilaciones con frecuencias distintas—
y ver qué trayectoria dibuja la punta. Es cinemática pura: dado cómo cambian
los ángulos, dónde está cada punto.

El puente es el de la semana 6: sustituir, `lambdify`, graficar.

In [ ]:
movimiento = {
    theta1: sp.pi/6 * sp.cos(t),
    theta2: sp.pi/3 * sp.cos(sp.sqrt(2)*t),
    L1: 1,
    L2: 1,
}

punta = P2.pos_from(O).express(N).to_matrix(N).subs(movimiento)
x_punta = sp.lambdify(t, punta[0], "numpy")
y_punta = sp.lambdify(t, punta[1], "numpy")

tiempos = np.linspace(0, 40, 2000)

fig, ax = plt.subplots()
ax.plot(x_punta(tiempos), y_punta(tiempos), linewidth=0.8)
ax.plot([0], [0], "ko")                        # el pivote
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Punta de un péndulo doble con movimiento impuesto")
plt.show()

Como $\sqrt{2}$ es irracional, las dos oscilaciones nunca vuelven a coincidir
y la curva va llenando una región sin cerrarse nunca. Nota el orden: primero
`subs` con el diccionario —que sustituye las funciones $\theta_i(t)$ por
funciones explícitas de $t$— y después `lambdify`, que solo sabe de números.

## Resumen

Hoy completamos la cinemática. Un `Point` se ubica respecto a otro con un
vector, y su velocidad y aceleración se piden **en un marco**; para eso algún
punto de la cadena necesita una velocidad conocida, o aparece el `ValueError`.
Un marco que gira con la partícula convierte las coordenadas polares en una
posición de un término, y de la derivada salieron solas las aceleraciones
centrípeta y de Coriolis.

Para cuerpos rígidos, `v2pt_theory` y `a2pt_theory` relacionan dos puntos fijos
en el mismo cuerpo, y `v1pt_theory` agrega el movimiento de un punto que se
desplaza sobre él. Encadenados, dan la cinemática del péndulo doble.

**Tarea de la semana:** [`tarea/tarea-07.ipynb`](../tarea/tarea-07.ipynb) —
la cinemática de una cuenta en un aro que gira.

**Próxima semana — Semana 8:** con las velocidades de hoy armaremos energías
cinéticas y Lagrangianos, y `mechanics` generará las **ecuaciones de
movimiento** por nosotros con `LagrangesMethod` y `KanesMethod`. Ahí el péndulo
doble deja de moverse como le imponemos y empieza a moverse como la física
manda.